In [ ]:
from copy import deepcopy
import glob
import json
import os

import pandas as pd

### Import data model

In [ ]:
data_model_df = pd.read_excel("../data/external/Data Model - Draft - for SI Union List - Populated.xlsx", sheet_name="Half Inch")
notes = data_model_df.iloc[0, :].to_dict()
data_model_df = data_model_df.drop(index=0).reset_index(drop=True)

In [ ]:
data_model_df.head(12)

In [ ]:
data_model_df.info()

### Import csv and metadata files

In [ ]:
csv_files = glob.glob("../data/interim/Half Inch/*.csv")
metadata_files = glob.glob("../data/interim/Half Inch/*.json")
# docx_files = [x for x in docx_files if "~" not in x and "(2)" not in x]
# docx_files = [x for x in docx_files if "_mod" not in x]

In [ ]:
csv_files, metadata_files

In [ ]:
dfs, metadatas = {}, {}
for f in csv_files:
    file_id = os.path.basename(f).split(".")[0]
    df = pd.read_csv(f, encoding="utf8")
    with open(f[:-4] + ".json") as g:
        metadata = json.load(g)
    dfs[file_id] = df
    metadatas[file_id] = metadata

In [ ]:
dfs["38B"].head(10)

In [ ]:
def process_row(row: pd.Series, source: str, scale: str) -> list[dict[str, str]]:
    """Process a map row into the current data model

    Args:
        row (pd.Series): A row from a .doc/.xlsx file with information about a map sheet
        source (str): The filename the row came from
        scale (str): One of "One Inch", "Half Inch", "Quarter Inch". The scale of the map as a string.
    """
    entries = []
    entry_template = {
        "Source File": source,
        "Series Title": f"Survey of India India and Adjacent Countries {scale} Series",
        "Scale": {"Quarter Inch": "1:253,440", "Half Inch": "1:126,720", "One Inch": "63,360"}[scale],
        "Published": None,  # How to tell if published and we don"t have a copy?
        "Location Room": "UGF",
        "Location Section": None,  # dict lookup with external resource, out of scope currently, see Issue #2
        "Location Detail": None,  # dict lookup with external resource, out of scope currently, see Issue #2
        "Full Reference": None,
        "Time Period": None,
        "Parent Reference": None,
        "Post-1905 Related References": None, # Introduced with Issue #3
        "1886-1905 Related References": None, # Introduced with Issue #3
        "Pre-1886 Related References": None, # Introduced with Issue #3
        "Post-1905 Block Number": None,
        "Post-1905 Block Letter": None,
        "Post-1905 Sheet ID": None,
        "1886-1905 New Sheet ID": None,
        "1886-1905 Old Sheet ID": None,
        "Pre-1886 New Sheet ID": None,
        "Pre-1886 Old Sheet ID": None,
        "Related Sheet": None,
        "Sheet Title": None,
        "Edition Number": None,
        "Edition Date": None,
        "Designation_1": None,
        "Designation_2": None,
        "Publication Date": None,
        "Print Date": None,
        "Print Reference": None,
        "Copies Printed": None,
        "Coloured": None,
        "Gridded": None,
        "Number of Copies": None,
        "Repmat": None,
        "Latitude": None,
        "Longitude": None,
        "Available": None,
        "Notes": ""
    }
    
    # Post-1905
    bn, bl = row.loc["Post-1905_1"].split("/")[0][:2], row.loc["Post-1905_1"].split("/")[0][2]
    si = row.loc["Post-1905_1"].split("/")[1]
    entry_template["Post-1905 Block Number"] = bn
    entry_template["Post-1905 Block Letter"] = bl
    entry_template["Post-1905 Sheet ID"] = si

    periods = ["Post-1905", "1886-1905", "Pre-1886"]

    # 1886-1905/Pre-1886 new/old sheet IDS
    for period in periods[1:]:
        col_1 = period + "_1"
        if not pd.isna(row.loc[col_1]):
            if len(row.loc[col_1].split("\n")) == 1:
                entry_template[f"{period} New Sheet ID"] = row.loc[col_1]
            elif len(row.loc[col_1].split("\n")) == 2:
                entry_template[f"{period} New Sheet ID"] = row.loc[col_1].split("\n")[1]
                entry_template[f"{period} Old Sheet ID"] = row.loc[col_1].split("\n")[0]

    
    # References
    references = {}
    for period in periods:
        col_2 = period + "_2"
        if col_2 in row.index and not pd.isna(row.loc[col_2]):
            lines = row.loc[col_2].split("\n")
            [references.update({l: period}) for l in lines if "X/" in l]

    if not references:
        entry = entry_template
        entries.append(entry)
        return entries
    
    time_period = {"Post-1905": "1905>", "1886-1905": "1886-1905", "Pre-1886": "<1886"}
    for ref, period in references.items():
        entry = deepcopy(entry_template)
        entry["Time Period"] = time_period[period]
        entry["Published"] = "Y"

        # Year and Reference
        x_num, year = ref.rsplit()
        entry["Print Date"] = year
        if "/" in year:
            year = year.split("/")[1]
            entry["Print Date"] = year
            entry["Notes"] += f"Print Date: {year}. Reference in source {source} indicates earlier Edition Date: {ref}\n"

        entry["Full Reference"] = "IOR/" + x_num
        entry["Parent Reference"] = "/".join(entry["Full Reference"].split("/")[:3])

        # Related References
        related_references = {"Post-1905 Related References": "", "1886-1905 Related References": "", "Pre-1886 Related References": ""}
        other_refs = references.copy()
        del other_refs[ref]

        for ref, period in other_refs.items():
            related_references[period + " Related References"] += ref + "\n"

        # Reference Notes
        lines = row.loc[f"{period}_2"].split("\n")
        if len(lines) > lines.index(ref) + 2 and "X/" not in lines[lines.index(ref) + 1]:
            entry["Notes"] += f"Reference coverage note: {lines[lines.index(ref) + 1]}\n"
                        
        entries.append(entry)

    return entries

In [ ]:
entries = []
[entries.extend(process_row(row[1], source="38B.doc", scale="Half Inch")) for row in dfs["38B"].iterrows()];

In [ ]:
entries[0]

In [ ]:
# This sort_values arranges Half Inch quadrants in the current BL cataloguing order of NW/NE/SW/SE compared to the previous standard of NW/SW/NE/SE
# I haven't implemented it as it makes it harder to map results to the raw data by row

# .sort_values(by=["Post-1905 Block Number", "Post-1905 Block Letter", "Post-1905 Sheet ID"], key=lambda x: x.apply(lambda y:{"NW":1, "SW":3, "NE":2, "SE":4}.get(y, y)))

In [ ]:
data_sample_df = pd.concat([pd.DataFrame(x, index=[0]) for x in entries]).reset_index(drop=True)
data_sample_df.to_csv("../data/processed/v0.2_sample.csv", encoding="utf8", index=False)
data_sample_df.iloc[:,:20]

In [ ]:
sources_by_scale = {
    "63,360": None,
    "1:126,720": set([os.path.basename(x) for x in glob.glob("C:\\Users\\hlloyd\\projects\\union-lists\\data\\raw\\Half Inch\\*.doc")]),
    "1:253,440": None
}

In [ ]:
assert len(data_sample_df["Scale"].unique()) == 1
assert set(data_sample_df["Source File"].unique()) <= sources_by_scale[data_sample_df["Scale"].unique()[0]]

        "Series Title": f"Survey of India India and Adjacent Countries {scale} Series",
        "Scale": {"Quarter Inch": "1:253,440", "Half Inch": "1:126,720", "One Inch": "63,360"}[scale],
        "Published": None,  # How to tell if published and we don"t have a copy?
        "Location Room": "UGF",
        "Location Section": None,  # dict lookup with external resource, out of scope currently, see Issue #2
        "Location Detail": None,  # dict lookup with external resource, out of scope currently, see Issue #2
        "Full Reference": None,
        "Time Period": None,
        "Parent Reference": None,
        "Post-1905 Block Number": None,
        "Post-1905 Block Letter": None,
        "Post-1905 Sheet ID": None,
        "1886-1905 New Sheet ID": None,
        "1886-1905 New Reference": None,
        "1886-1905 Old Sheet ID": None,
        "1886-1905 Old Reference": None,
        "Pre-1886 New Sheet ID": None,
        "Pre-1886 New Reference": None,
        "Pre-1886 Old Sheet ID": None,
        "Pre-1886 Old Reference": None,
        "Related Sheet": None,
        "Sheet Title": None,
        "Edition Number": None,
        "Edition Date": None,
        "Designation_1": None,
        "Designation_2": None,
        "Publication Date": None,
        "Print Date": None,
        "Print Reference": None,
        "Copies Printed": None,
        "Coloured": None,
        "Gridded": None,
        "Number of Copies": None,
        "Repmat": None,
        "Latitude": None,
        "Longitude": None,
        "Available": None,
        "Notes": ""

In [ ]:
data_sample_df.loc[0, "1886-1905 Old Sheet ID"]

Design decisions
 - All locations are UGF, the Section/Detail to be added cross-referencing other sources at a later date
 - All references contain "X/"
 - All references produce a separate row in the data format
 - Time period is set by the period a reference comes from
 - If the final part of a reference contains multiple dates separated by a "/" then the second date is taken as the print year and this text is added to Notes: "Print Date: {year}. Reference in source {source} indicates earlier Edition Date: {reference}"
 - Any line after a reference that does not contain "X/" is a note about the preceding reference and the following text is added to Notes: f"Reference coverage note: {reference}\n"

In [ ]:
dfs["38B"].head(20)

### Work using old Maps IS file

In [ ]:
is_dtype = {"Block Number": "Int64", "Sheet Number": "Int64", "Date.1": "str", "Date.2": "str", "Date.3": "str", "Drawer": "Int64"}
is_df = pd.read_excel("../data/raw/Maps IS.xlsx", sheet_name="54 RAW", skiprows=5, header=None, names=["Block Number", "Block Letter", "Sheet Number", "Date.1", "Date.2", "Date.3", "Drawer", "Shelfmark", "Extra Date.1", "Extra Date.2"], dtype=is_dtype)
is_df.drop(index=0, inplace=True)
is_gt_df = pd.read_excel("../data/raw/Maps IS.xlsx", sheet_name="54 NEW")

x9_df = pd.read_excel("../data/raw/X_9053.xlsx", sheet_name="54 Raw", skiprows=6, skipfooter=4, header=None, usecols=[0,1], names=["Block String", "Full Reference"])
x9_gt_df = pd.read_excel("../data/raw/X_9053.xlsx", sheet_name="54 New")

In [ ]:
data_model_df.columns

In [ ]:
is_df.columns

In [ ]:
include_if_present = ["Date.2", "Date.3", "Extra Date.1", "Extra Date.2"]
by_date = [is_df.drop(columns=include_if_present).rename(columns={"Date.1": "Date"})]
for col in include_if_present:
    by_date.append(is_df[["Block Number", "Block Letter", "Sheet Number", "Drawer", "Shelfmark"] + [col]].dropna(subset=col).rename(columns={col: "Date"}))

In [ ]:
clean_is_df = pd.concat(by_date).sort_values(["Block Number", "Block Letter", "Sheet Number"]).reset_index(drop=True)
clean_is_df["Village Boundaries"] = clean_is_df["Date"].str.contains("vb")
clean_is_df["Date"] = clean_is_df["Date"].str.strip(" vb").str.strip("&")

In [ ]:
clean_is_df

In [ ]:
expected_output_length = is_df.shape[0] + is_df.dropna(subset="Date.2").shape[0] + is_df.dropna(subset="Date.3").shape[0] + is_df.dropna(subset="Extra Date.1").shape[0] + is_df.dropna(subset="Extra Date.2").shape[0]

In [ ]:
assert expected_output_length == clean_is_df.shape[0]
assert expected_output_length == is_gt_df.shape[0]

In [ ]:
is_gt_df.head()

In [ ]:
x9_df.head()

In [ ]:
x9_df.shape

In [ ]:
pd.isna(x9_df.head().loc[2, "Full Reference"])

In [ ]:
def split_block(s: str|pd.NA) -> (str, str, str)|(pd.NA, pd.NA, pd.NA):
    if pd.isna(s):
        return (pd.NA, pd.NA, pd.NA)
        
    n = s.split("/")[0][:-1]
    l = s.split("/")[0][-1]
    sn = s.split("/")[1]
    return (n, l, sn)

In [ ]:
def split_ref(s: str|pd.NA) -> str|pd.NA:
    if pd.isna(s):
        return pd.NA

    date = s.split()[-1]
    return date

In [ ]:
clean_x9_df = pd.DataFrame(columns=data_model_df.columns)
clean_x9_df["Full Reference"] = x9_df["Full Reference"]
x9_df["Block Split"] = x9_df["Block String"].apply(lambda x: split_block(x))
clean_x9_df["Block Number"] = x9_df["Block Split"].apply(lambda x: x[0])
clean_x9_df["Block Letter"] = x9_df["Block Split"].apply(lambda x: x[1])
clean_x9_df["Sheet Number"] = x9_df["Block Split"].apply(lambda x: x[2])
clean_x9_df["Publication Date"] = clean_x9_df["Full Reference"].apply(lambda x: split_ref(x))

In [ ]:
clean_x9_df.head()

In [ ]:
x9_gt_df.head()